# TPU v5e-8: 8K midtrain -> 16K SFT

Select **TPU v5e-8**, enable Internet, attach the prepared posttrain corpus and the
**completed 2.4B pretrain checkpoint**. Set their paths below. This notebook runs
both remaining stages in order, with no new model initialization for training.

1. **Midtrain:** 600M nonpadding tokens, 8K rows, 80/5/15 document/reasoning/agent.
   Restores pretrained parameters **and optimizer**, preserving the original 3B LR
   horizon and global update counter. Candidate masking stays off.
2. **SFT:** 16K rows, 1:2 reasoning/agent, assistant-only targets. Resets optimizer
   and uses a new low-LR schedule. Enables compressed-layer YaRN with original
   length 8192 / factor 2 and the planned SFT candidate mask. Pure local RoPE stays
   unchanged. `SFT_TOKENS=0` computes a one-pass budget from actual pool capacity;
   it does not mean train zero tokens or concatenate every source indiscriminately.

The 16K workload has **not yet been measured on your TPU**. First compilation and
update validate the native path, loss-mask count, finite loss and zero dropped
MoE assignments. A checkpoint is saved before compilation and after the first
successful update. There is no automatic fallback to 8K if 16K cannot compile.

An eight-hour deadline includes setup, leaving margin before Kaggle's nine-hour
limit. The runner pauses safely between updates and resumes the current stage,
shuffle cursors and optimizer. Both stages may require more than one session.

In [ ]:
import os
import time
SESSION_STARTED = time.time()
SESSION_HOURS = 8.0
CORPUS = '/kaggle/input/your-posttrain-corpus-dataset'
PRETRAINED_CHECKPOINT = '/kaggle/input/your-pretrain-output/training/checkpoints'
RESUME_CHECKPOINT = ''  # later sessions: current midtrain/sft checkpoint directory
SFT_TOKENS = 0  # derive one-pass budget at fixed 1:2 ratio; keep unchanged on resume
SFT_LR = 2.6e-5
os.environ['NANO_DSV41F_REF'] = 'codex/midtrain8k-sft16k'

In [ ]:
"""Canonical notebook bootstrap. Standard library only: runs before importing JAX."""
from pathlib import Path
from importlib.metadata import version
import os
import subprocess
import sys

if any(name in sys.modules for name in ("jax", "jaxlib", "libtpu", "tokamax", "nano_dsv41f")):
    raise RuntimeError("Restart the Kaggle session and run this cell first: JAX/TPU was already imported.")

ROOT = Path('/kaggle/working/nano-dsv4.1f')
REPO_REF = os.environ.get('NANO_DSV41F_REF', 'main')
if not ROOT.exists():
    subprocess.run(['git', 'clone', '--depth', '1',
                    'https://github.com/xiayicheng3-code/nano-dsv4.1f.git', str(ROOT)], check=True)
if not (ROOT / '.git').exists():
    raise RuntimeError(f'{ROOT} exists but is not a git checkout.')
status = subprocess.check_output(['git', '-C', str(ROOT), 'status', '--porcelain',
                                  '--untracked-files=no'], text=True)
if status.strip():
    raise RuntimeError('Checkout has tracked edits; preserve them before updating:\n' + status)
subprocess.run(['git', '-C', str(ROOT), 'fetch', '--depth', '1', 'origin', REPO_REF], check=True)
subprocess.run(['git', '-C', str(ROOT), 'checkout', '--detach', 'FETCH_HEAD'], check=True)
# Pin the entire validated package combination, not just the Python jax package.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade',
                '-r', str(ROOT / 'requirements-tpu.txt')], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(ROOT), '--no-deps'], check=True)
os.chdir(ROOT)
os.environ['PYTHONPATH'] = str(ROOT / 'src') + os.pathsep + os.environ.get('PYTHONPATH', '')
os.environ['JAX_PLATFORMS'] = 'tpu'
print('repo:', ROOT)
print('requested ref:', REPO_REF)
print('commit:', subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip())
print('packages:', {p: version(p) for p in ('jax', 'jaxlib', 'libtpu', 'tokamax')})
# Execution cells run fresh Python processes. Never retain an old TPU client in IPython.


In [ ]:
import json
from datetime import datetime,timezone
if not Path(CORPUS).is_dir(): raise FileNotFoundError('Set CORPUS to the attached CPU notebook output')
checkpoint = RESUME_CHECKPOINT or PRETRAINED_CHECKPOINT
if not Path(checkpoint).is_dir(): raise FileNotFoundError('Set the checkpoint path')
RUN_ROOT = Path('/kaggle/working') / ('posttrain-'+datetime.now(timezone.utc).strftime('%Y%m%d-%H%M%S'))
RUN_ROOT.mkdir(exist_ok=False)
OUTPUT = RUN_ROOT / 'training'
subprocess.run(['git','archive','--format=tar.gz','-o',str(RUN_ROOT/'source.tar.gz'),'HEAD'],check=True)
command = [sys.executable,'-u','scripts/run_posttrain.py',
    '--corpus',CORPUS,'--output',str(OUTPUT),
    '--resume' if RESUME_CHECKPOINT else '--pretrained',checkpoint,
    '--sft-tokens',str(SFT_TOKENS),'--sft-lr',str(SFT_LR),
    '--deadline-unix',str(SESSION_STARTED+SESSION_HOURS*3600)]
with (RUN_ROOT/'train.log').open('w',buffering=1) as log:
    p = subprocess.Popen(command,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    try:
        for line in p.stdout:
            print(line,end='',flush=True); log.write(line)
        rc = p.wait()
    except KeyboardInterrupt:
        p.terminate()
        print('Waiting for the runner to checkpoint between updates')
        rc = p.wait()
summary = json.loads((OUTPUT/'summary.json').read_text()) if (OUTPUT/'summary.json').exists() else {}
print(json.dumps(summary,indent=2))
if rc: raise RuntimeError('Training failed; inspect train.log. The previous committed checkpoint is retained.')

## Resume / completion

Save these outputs as a Kaggle dataset. `summary.json` reports `paused`, `completed`
or `failed` and the exact latest checkpoint path. On a fresh session, attach the
saved output, set `RESUME_CHECKPOINT` to that directory (or its parent containing
`latest.json`), and keep the source version, corpus, SFT budget, LR and seed unchanged.
The source commit is printed by bootstrap and archived; pin `NANO_DSV41F_REF` to
that commit for future resumes if the branch moves.

Midtrain and SFT checkpoints have separate directories. `completed` means both
stages finished. Each checkpoint includes the effective model/YaRN configuration,
full optimizer state and data cursor. Keep the final SFT recipe with the weights
for inference. QAT and DSpark retain the existing recipe settings; this notebook
does not introduce an additional training stage.

In [ ]:
from IPython.display import FileLink,display
for name in ('summary.json','metrics.jsonl','launch.json'):
    path = OUTPUT/name
    if path.exists(): display(FileLink(str(path)))
display(FileLink(str(RUN_ROOT/'train.log')))
print('Latest checkpoint:', summary.get('checkpoint','No committed checkpoint yet'))
print('Save Version outputs to preserve checkpoint directories and source.tar.gz.')